<a href="https://colab.research.google.com/github/awkaningg/UTS-DAM/blob/main/API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import json
import pandas as pd
import html

# 1. Akuisisi Data dari WordPress REST API Museum Nasional (100 data terbaru)
url_api = "https://museumnasional.or.id/wp-json/wp/v2/posts"
headers = {"User-Agent": "Mozilla/5.0 (UTS-DAM Academic Project)"}
params = {"per_page": 100}

response = requests.get(url_api, headers=headers, params=params)
print(f"Status HTTP: {response.status_code}")

data_raw = response.json()
print(f"Total data mentah diambil: {len(data_raw)} artikel")

# Simpan data mentah JSON
with open("api_museum_raw.json", "w", encoding="utf-8") as f:
    json.dump(data_raw, f, indent=4, ensure_ascii=False)

# 2. Ekstraksi & Minimalisasi Kolom
records = []
for item in data_raw:
    records.append({
        "ID": item.get("id"),
        "Tanggal_Terbit": item.get("date"),
        "Judul": html.unescape(item.get("title", {}).get("rendered", "")).strip(),
        "URL": item.get("link", "").strip()
    })

df_api = pd.DataFrame(records)

# 3. Pembersihan & Standardisasi Format Tanggal
df_api = df_api.dropna().drop_duplicates(subset=["ID"])
df_api["Tanggal_Terbit"] = pd.to_datetime(df_api["Tanggal_Terbit"]).dt.strftime("%Y-%m-%d %H:%M:%S")

# 4. Pengecekan PII & Qualifier
pola_pii = r"(?:[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})|(?:(?:\+62|62|0)8[1-9][0-9]{6,10})"
jumlah_pii = df_api["URL"].str.contains(pola_pii, regex=True).sum()
print(f"Jumlah temuan PII pada URL: {jumlah_pii}")

df_api["Tahun"] = pd.to_datetime(df_api["Tanggal_Terbit"]).dt.year
print("\nDistribusi Artikel per Tahun (Qualifier):")
print(df_api["Tahun"].value_counts())

# Hapus kolom bantu Tahun agar struktur kolom tetap bersih
df_api = df_api.drop(columns=["Tahun"])

# 5. Simpan hasil bersih ke CSV
df_api.to_csv("api_museum_process.csv", index=False)
print("\nBerhasil! File 'api_museum_process.csv' telah dibuat.")
display(df_api.head(10))

Status HTTP: 200
Total data mentah diambil: 100 artikel
Jumlah temuan PII pada URL: 0

Distribusi Artikel per Tahun (Qualifier):
Tahun
2022    55
2026    14
2023    14
2025     9
2024     8
Name: count, dtype: int64

Berhasil! File 'api_museum_process.csv' telah dibuat.


,ID,Tanggal_Terbit,Judul,URL
0,5524,2026-10-23 12:47:07,Wayang Kulit Gaya Yogyakarta: Tokoh Mahabharata,https://www.museumnasional.or.id/5524/
1,5521,2026-10-23 12:43:14,"Wayang: Merentang Masa, Merawat Makna",https://www.museumnasional.or.id/5521/
2,5490,2026-09-18 18:17:50,Jelajah Konservasi: Mengenal Proses di Balik P...,https://www.museumnasional.or.id/5490/
3,5481,2026-08-05 09:45:48,Museum Nasional Indonesia Kembali Gelar Bincan...,https://www.museumnasional.or.id/5481/
4,5469,2026-03-28 15:57:21,Museum Nasional Indonesia Gelar Talkshow “Keti...,https://www.museumnasional.or.id/5469/
5,5462,2026-03-10 16:49:26,Kebersamaan Ramadan di Museum Nasional Indones...,https://www.museumnasional.or.id/5462/
6,5449,2026-03-03 13:20:39,Semarak Ramadan di Museum Nasional Indonesia: ...,https://www.museumnasional.or.id/5449/
7,5435,2026-02-13 13:20:50,Google Arts & Culture Launches the Digital Pla...,https://www.museumnasional.or.id/5435/
8,5431,2026-02-13 13:19:21,Google Arts & Culture Luncurkan Platform Digit...,https://www.museumnasional.or.id/5431/
9,5427,2026-02-04 09:23:14,Menikmati Rabu Sore Bebas Biaya di Museum bers...,https://www.museumnasional.or.id/5427/
